# PriorModel — Colab setup

This notebook installs **Julia 1.12.4**, clones the repo, mounts Drive, instantiates the project, and fetches or builds training data.

**Before anything else:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is enough).

Keep the kernel as **Python 3**. Julia is invoked with `!julia --project=/content/PriorModel ...` so we never depend on Colab's built-in Julia 1.10 runtime.

## Do not use `--project=.`

Colab's working directory is `/content`. Running `julia --project=.` from there creates a leftover **`/content/Project.toml`** that hides this repository's environment. After that, `using HDF5` and `using LuxCUDA` fail with "package not found in current path".

Always pass the clone path:

```bash
julia --project=/content/PriorModel …
```

If `/content/Project.toml` already exists, delete it in the next cell.

In [ ]:
%%bash
set -euo pipefail
if [[ -f /content/Project.toml ]]; then
  echo "Removing leftover /content/Project.toml (created by --project=.)"
  rm -f /content/Project.toml /content/Manifest.toml
fi
ls -la /content/Project.toml 2>/dev/null || echo "No /content/Project.toml — good."

## Install Julia 1.12.4

In [ ]:
%%bash
set -euo pipefail
JULIA_VERSION="1.12.4"
JULIA_VER="${JULIA_VERSION%.*}"
if julia --version 2>/dev/null | grep -q "${JULIA_VERSION}"; then
  julia --version
  exit 0
fi
echo "Installing Julia ${JULIA_VERSION}…"
URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_VER}/julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
wget -q "${URL}" -O /tmp/julia.tar.gz
tar -xzf /tmp/julia.tar.gz -C /usr/local --strip-components=1
rm /tmp/julia.tar.gz
julia --version

## Clone the repository

In [ ]:
%%bash
set -euo pipefail
# Public repo: never wait for GitHub username/password (that hangs this cell).
export GIT_TERMINAL_PROMPT=0
export GIT_PAGER=cat
unset GIT_ASKPASS SSH_ASKPASS

REPO_URL="https://github.com/hayrunnisayildiz/PriorModel.git"
ZIP_URL="https://github.com/hayrunnisayildiz/PriorModel/archive/refs/heads/master.zip"
DEST="/content/PriorModel"

echo "=== public clone (no login) ==="

if [[ -d "${DEST}/.git" ]]; then
  echo "Updating existing clone…"
  git -C "${DEST}" -c credential.helper= --no-pager fetch --depth 1 origin master
  git -C "${DEST}" --no-pager reset --hard FETCH_HEAD
else
  rm -rf "${DEST}"
  echo "Cloning ${REPO_URL} …"
  if timeout 90 git -c credential.helper= clone --depth 1 --single-branch --branch master --progress "${REPO_URL}" "${DEST}"; then
    echo "git clone OK"
  else
    echo "git clone stalled/failed — downloading public zip instead"
    rm -rf "${DEST}"
    wget -q --show-progress -O /tmp/PriorModel.zip "${ZIP_URL}"
    unzip -qo /tmp/PriorModel.zip -d /tmp
    mv /tmp/PriorModel-master "${DEST}"
    rm -f /tmp/PriorModel.zip
  fi
fi

test -f "${DEST}/Project.toml"
ls -l "${DEST}/Project.toml"
echo "OK"

## Mount Google Drive

HDF5 files are under this nested folder:

`/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel/data/synthetic/`

- `train_pairs.h5` — small smoke test (~5 MB)
- `train_pairs_v7.h5` — production set (~28 MB)

Those paths are gitignored (see `.gitignore`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Instantiate the Julia project

`MTGeophysics.jl` is pulled from GitHub tag **v0.4.2** (`Project.toml` `[sources]`), not from a local path.

**GLMakie warning:** MTGeophysics lists GLMakie as a hard dependency and imports it at package load. Instantiate still downloads it. On this headless VM, `src/pkg_setup.jl` skips auto-precompile so instantiate does not open an OpenGL context. Do **not** `using MTGeophysics` here; training must use `--commemi-every 0` (see last cell).

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
echo "=== Pkg.instantiate (several minutes, GLMakie downloads but must not precompile) ==="
julia --project=/content/PriorModel -e '
println("julia started; instantiating…"); flush(stdout); flush(stderr)
using Pkg
Pkg.instantiate()
println("active=", Base.active_project())
flush(stdout)
'

## Fetch data from Drive

Artifacts live under the nested Drive folder (not `MyDrive/PriorModel/…`):

`MyDrive/PriorModelData/PriorModel/PriorModel/data/synthetic/`

The next cell copies `train_pairs.h5` (small) and `train_pairs_v7.h5` (production). Training should use **v7**.

In [ ]:
from pathlib import Path
import shutil

DRIVE_SYN = Path(
    "/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel/data/synthetic"
)
DST_DIR = Path("/content/PriorModel/data/synthetic")
NAMES = ("train_pairs.h5", "train_pairs_v7.h5")

print("=== copy from Drive ===", flush=True)
print("source:", DRIVE_SYN, "exists=", DRIVE_SYN.is_dir(), flush=True)
if not DRIVE_SYN.is_dir():
    raise FileNotFoundError(f"Drive synthetic folder missing: {DRIVE_SYN}")

DST_DIR.mkdir(parents=True, exist_ok=True)
for name in NAMES:
    src = DRIVE_SYN / name
    dst = DST_DIR / name
    if not src.is_file():
        print(f"skip (missing): {src}", flush=True)
        continue
    print(f"copying {name} ({src.stat().st_size / 1e6:.1f} MB) …", flush=True)
    shutil.copy2(src, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.1f} MB)", flush=True)

v7 = DST_DIR / "train_pairs_v7.h5"
print("\ntrain with:", v7 if v7.is_file() else DST_DIR / "train_pairs.h5", flush=True)

## Train

On Colab you **must** pass `--commemi-every 0` and `--no-plot`.

Use `train_pairs_v7.h5` (n≈1000-scale production set). The 5 MB `train_pairs.h5` is only a small smoke-test file.

MTGeophysics.jl cannot be used without loading GLMakie. The COMMEMI probe therefore stays off on this headless VM. `--no-plot` skips the training-curve PNG.

Always pass `--project=/content/PriorModel` (never `--project=.`).

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
julia --project=/content/PriorModel \
  /content/PriorModel/src/training/train_mt_resistivity.jl \
  --dataset /content/PriorModel/data/synthetic/train_pairs_v7.h5 \
  --commemi-every 0 --no-plot